# Lesson 70 · Inside Whisper — taking apart a machine that understands 99 languages: log-mel → encoder–decoder → multitask heads

**Goal**: first understand Whisper's four-act architecture, then set it side by side with the Aurora DSP and CTC route.

> **All this lesson asks of you**: ① draw the four-box diagram (log-mel → encoder → decoder → tokens)
> ② name one way it differs from CTC (L68–69): autoregressive generation vs summing over alignments.
> Treat the paper's hyperparameters and ablations as "further reading"; the main path is kept lean.

🔗 **Aurora connection**: the 80-dim log-mel is L46–L47's `mel_spectrogram(n_mels=80)` (about a 25 ms window / 10 ms hop).

← **Previous lesson** · [L69 · The CTC forward algorithm](L69_ctc_forward.ipynb)

> Last lesson covered **the CTC forward algorithm**: summing the probability of every path, O(T·S) (S=2L+1), in pure NumPy.
> This lesson explores **inside Whisper**.

## Learning goals

By the end of this lesson you should be able to:

1. Understand how Whisper's 80-dim log-mel input is produced and how it lines up with Aurora DSP's parameters (`n_fft=400, hop=160`)
2. Describe the encoder's chain of shape changes through Conv1d×2 + Transformer (input 3000 → conv downsampling → 1500 → Transformer → (1500, d_model))
3. Explain why, in cross-attention, Q comes from the decoder while K/V come from the encoder, and why the attention weights have shape `(B, T_dec, T_enc)`
4. List the four outputs of Whisper's multitask head and the conditioning token-prefix mechanism
5. Implement `whisper_preprocess`, passing three sets of assertions: 30 s of silence, 5 s of short audio zero-padded, and 60 s of long audio truncated

## Whisper / CTC connection note

Whisper is trained with cross-entropy, not CTC; L68–L69 were the stop before this one, for understanding "the sequence-alignment baseline".
Understand CTC's monotonic paths and log-domain DP first, and when you get to Whisper you'll see why it chose token-by-token autoregressive decoding.

## This lesson's story: how can Whisper "understand 99 languages"?

Whisper, released by OpenAI in 2022, transcribes speech in 99 languages without any special engineering. The secret isn't "magic" — it's data, architecture and training objective stacked up together.

**The answer: massive weak supervision + encoder–decoder + multitask heads.**

Whisper squeezes the whole ASR pipeline into one architecture:

```
audio (16kHz)
   ↓ log-mel (n_fft=400, hop=160, n_mels=80, 30s→3000 frames)
   ↓ 2×Conv1D downsampling → a sequence of 1500 vectors
   ↓ Transformer encoder (many layers of self-attention)
   ↓ context vectors (B, 1500, d_model)
   ↓ Transformer decoder (cross-attention to the encoder)
   ↓ BPE token probabilities (B, T_dec, vocab_size)
   ↓ text
```

**Three key design choices**:

| Design | Purpose |
|---|---|
| A fixed 30-second window (3000 frames) | One input shape; no variable-length handling needed |
| 2×Conv1D downsampling (the second with stride=2) | 3000 frames → 1500 steps, cutting the cost of attention over the audio |
| Multitask tokens (`<en>`,`<transcribe>`,`<translate>`) | One model does transcription / translation / voice-activity detection — no need for several models |

**Aurora connection**: `aurora.audio.mel.mel_spectrogram()` can be called with Whisper's official parameters (n_mels=80, n_fft=400, hop=160, sr=16000), so its frames line up one-to-one with Whisper's — this lesson's `whisper_preprocess()` calls it directly. (It is not bit-identical: by default it zero-pads the 400-point frame to a 512-point FFT and uses the HTK mel scale, while Whisper uses a true 400-point FFT and the Slaney mel scale. The exact recipe is `aurora.audio.whisper_frontend.log_mel_whisper` — see REPORT.md in the AURORA repository.)

This lesson's task: implement `whisper_preprocess(wav)` — audio → a `(1, 80, 3000)` tensor.

### Digging deeper: why exactly 30 seconds, and why exactly 1500 steps?

The table above says the "fixed 30-second window" is there to "unify the input shape", and "downsampling to 1500 steps" is there to "cut computation". But **why 30 seconds and not 20 or 40? Why 1500 and not 1000 or 2000?**

The conclusion first: **neither number is "uniquely derived" from some formula. Both are engineering trade-offs that OpenAI settled on after a lot of experiments** — a bit like a camera maker choosing a photo resolution: go a little higher and image quality barely improves while storage balloons; go a little lower and you can't make out the details. Below, we put numbers on what this "trade-off" is actually weighing up.

#### Trade-off 1: self-attention costs T squared, not T

In the Transformer encoder's self-attention, every step computes a similarity with **every other step** (section 3 covers this in detail). With a sequence of length T, the total number of "pairwise similarities" is:

```
cost ∝ T × T = T²
```

Which means: **halve the sequence length and the cost doesn't halve — it drops to 1/4!** That's exactly the answer to Samuel's follow-up question, "what concrete effect does downsampling by 2 have on the cost?": the code below computes the cost for T=3000 (no downsampling), 1500 (Whisper's actual choice) and 750 (downsampling once more) — 1500 steps cost only 1/4 of 3000 steps. That's the real payoff from "downsampling once".

#### Trade-off 2: the more you downsample, the coarser the time resolution, and phoneme detail can be lost

Each stride=2 downsampling amounts to "merging two frames into one step". After downsampling, each step spans:

```
duration of 1 step = hop(10ms) × total conv downsampling factor(2) = 20 ms
```

Downsample once more (4× in total) and each step has to cover 40 ms. The problem: the difference between consonants like "b" and "p" often lies in a burst lasting just tens of milliseconds — downsample too aggressively and these brief acoustic cues may get "averaged" into neighboring steps, making them harder for the model to tell apart. So Whisper's stride=2 (20 ms/step) is a trade-off point between "cost" and "time resolution" — more downsampling is not always better.

#### Trade-off 3: the 30-second window has to cover "how long a sentence usually is"

A longer audio window isn't automatically better — the longer the window, the bigger the T² cost from trade-off 1. Thirty seconds is enough to hold a whole sentence, or even a short passage, in most natural speech, without making T absurdly large. If you have a 5-minute recording, Whisper doesn't stuff it into the model all at once — it processes it in segments with a **sliding 30-second window** (with overlap) and stitches the results together (this is the long-form transcription strategy that comes up after L71). So "30 seconds" is more like "the size of one bite" than "a hard limit on sentence length".

**Exercise**: work it out yourself — what if Whisper had chosen a 20-second window? The code below has already computed the 20/30/40-second cases. Read the output and think about the trade-off between "how much compute you save" and "how much speech fits in".

In [ ]:
# Demo: how window length and downsampling factor affect the sequence length T and the self-attention cost (T²)
def window_stats(seconds, sr=16000, hop=160, downsample=2):
    n_samples = seconds * sr
    n_frames = n_samples // hop
    T = n_frames // downsample
    compute = T ** 2
    return n_frames, T, compute

print("Window length comparison (downsampling fixed at 2×):")
for seconds in [20, 30, 40]:
    n_frames, T, compute = window_stats(seconds)
    print(f"  {seconds:2d} s → mel frames={n_frames:5d} → after downsampling T={T:5d} → self-attention cost (T²)={compute:>10,}")

print("\nDownsampling factor comparison (30-second window fixed):")
baseline = None
for downsample in [1, 2, 4]:
    n_frames, T, compute = window_stats(30, downsample=downsample)
    baseline = compute if baseline is None else baseline
    print(f"  downsample {downsample}× → T={T:5d} → cost (T²)={compute:>10,}  (relative to no downsampling: {compute/baseline*100:.1f}%)")


In [ ]:
import torch
import numpy as np
from aurora.audio.mel import mel_spectrogram, power_to_db


## 1. Whisper's input: an 80-dim log-mel of shape `(80, 3000)`

### From "sample rate" to "frames": why hop=160?

The audio's sample rate is 16 kHz, meaning 16,000 numbers are recorded every second. Thirty seconds of audio is 16000 × 30 = **480,000** samples.

But we can't hand all 480,000 points to the model — far too dense. Think of a film: nobody needs a fresh picture every millisecond; 24 frames a second is plenty to follow the action.

By the same logic, we can "downsample" the audio like this:

```
extract features every 160 samples → one "frame"

30 seconds = 480,000 samples
480,000 ÷ 160 = 3,000 frames
```

This 160 is called the **hop** or **hop_length**. Why 160?
- At a 16 kHz sample rate, one second is 16,000 points
- 160 points last 160 / 16,000 = **0.01 s = 10 ms**
- So hop=160 means one frame every 10 ms

You can picture it as a "sliding window" (the diagram shows only where each frame starts; each window is actually 400 samples long and neighboring windows overlap):

```
samples:   [S0, S1, S2, ..., S159, S160, ..., S319, S320, ...]
window 1:  [S0  ~ S159]   → features for frame 1
window 2:          [S160 ~ S319]   → features for frame 2 (window moves on by 160 points)
window 3:                  [S320 ~ S479]   → features for frame 3
...
```

Run a short-time Fourier transform (STFT) inside each window to extract the frequency information, and you get the mel spectrogram.

**The core formula**:
```
number of frames = total samples / hop_length
                 = 480,000 / 160
                 = 3,000
```

---

Whisper always uses a **30-second** audio window (zero-padded if shorter, truncated if longer). At a 16 kHz sample rate, 30 s = 480 000 samples. STFT parameters: `n_fft=400` (a 25 ms window), `hop=160` (a 10 ms hop).

**Why these parameters?**

- **n_fft=400**: at 16 kHz, 400 points = a 25 ms time window — roughly the timescale at which we perceive phonemes.
- **hop=160**: that's n_fft / 2.5, i.e. one frame every 10 ms — dense enough to track pitch and phoneme boundaries.

The frame count is `480000 // 160 = 3000` (the official convention), and the mel dimension is fixed at 80. Note: Aurora's `mel_spectrogram` defaults to `center=True` (reflect padding) and actually returns **3001** frames (`480000 // 160 + 1`) — the official implementation likewise computes one extra frame and then drops the last one, and the truncation in exercise step 3 smooths over exactly this one-frame difference.

---

Whisper's log-mel normalization:

```
log_mel = log10(max(mel_power, 1e-10))
log_mel = max(log_mel, log_mel.max() - 8.0)
log_mel = (log_mel + 4.0) / 4.0
```

**What are these three lines doing?**

**Line 1**: `log10(max(mel_power, 1e-10))`
- Take the log of the band energies (log scale), compressing the dynamic range.
- `max(..., 1e-10)` prevents log(0) by giving tiny values a floor.

**Line 2**: `max(log_mel, log_mel.max() - 8.0)`
- Find the maximum of log_mel, then "clip" everything below `max - 8.0` up to that floor.
- Effect: **flatten outliers and limit the dynamic range**.
- Why subtract 8.0? On a log10 scale, 8.0 is an energy ratio of 10^8 (80 dB) — roughly the largest range the ear can make use of.
- For example, if log_mel.max() = 0 (a relative energy of 10^0 = 1), the clip floor is -8.0 (10^-8, an extremely faint sound). Anything below -8.0 (ultra-faint noise) is bluntly set to -8.0.

**Line 3**: `(log_mel + 4.0) / 4.0`
- A linear transform that scales the values to around `[-1, 1]`.
- Why add 4.0 and then divide by 4.0?
  - After line 2, log_mel lies roughly in `[clip floor, log_mel.max()]`, e.g. `[-8.0, 0]`.
  - Add 4.0: `[-8.0, 0]` → `[-4.0, 4.0]` (now the range is symmetric about 0)
  - Divide by 4.0: `[-4.0, 4.0]` → `[-1, 1]` (normalized to [-1, 1])
  - **Why do this?** Neural networks learn more easily from data in the [-1, 1] range (more stable gradients) than from raw log energies of [-80, 0] dB.

**Check**: for a loud recording whose loudest mel bin is at 0, the faintest bins are clipped to -8.0, so:
```
(-8.0 + 4.0) / 4.0 = -4.0 / 4.0 = -1.0  ✓
```
and the highest-energy mel bin:
```
(0 + 4.0) / 4.0 = 1.0  ✓
```
Pure digital silence is a special case: every bin is log10(1e-10) = -10, so the maximum is also -10, the clip changes nothing, and every value becomes (-10 + 4.0) / 4.0 = -1.5 — which is exactly what the check cell below prints.



In [ ]:
# Demo: check the frame-count formula
sr = 16000
n_seconds = 30
hop = 160
n_samples = sr * n_seconds
n_frames = n_samples // hop
print(f"30 s @ 16 kHz = {n_samples} samples")
print(f"hop={hop} → frames = {n_samples} / {hop} = {n_frames}")
print(f"Whisper input shape: (80, {n_frames})")


### Why does Aurora's mel_spectrogram return 3001 frames instead of 3000?

You may have noticed: 30 seconds of audio should in theory be 3000 frames, but Aurora (and official Whisper) actually produce **3001**. Where does the extra frame come from?

**The reason: center=True reflection padding**

When doing an STFT, the edges are a problem:

```
samples [S0, S1, S2, ..., S_479999]
         ↑ we want a window centered here, but there is no data "before" the first sample
```

With center=True, frame k is meant to be *centered* on sample k × 160. For frame 0 that means a 400-point window centered on S0: [?, ?, ..., ?, S0, S1, ..., S199] — the first 200 points are missing.

**center=True** solves this by adding data at the start and the end, so that a full window can be centered on S0. Adding what? **Reflection padding (reflect)**:

```
original:                  [S0, S1, S2, ..., S_479999]
200 points added in front: [S200, S199, ..., S1,  S0, S1, S2, ..., S_479999]
                            ↑ a mirror image, as if the audio "bounced back" off the edge

200 points added behind:   [S200, ..., S0, ..., S_479999, S_479998, S_479997, ..., S_479799]
                                                            ↑ mirrored the same way
```

With these 200+200 = 400 padding points, there are 200+480000+200 = 480400 points in total. Now frame it with a 400-point window and hop=160:

```
1 + (480400 - 400) // 160 = 1 + 3000 = 3001 frames
```

In other words, there is one frame centered on every multiple of 160 from sample 0 up to *and including* sample 480000 — that's 3001 centers. The last one (frame 3001) sits right at the very end of the audio and is mostly padding, carrying almost no real audio information, so in the exercise you need to **truncate to 3000**.

**The core code logic**:
```python
mel_raw = mel_spectrogram(wav, center=True)  # returns (3001, 80)
mel_3000 = mel_raw[:3000]  # drop the last frame → (3000, 80)
```

That's why step 3 of `whisper_preprocess` has to "pad or truncate to 3000 frames".



## 2. The encoder: Conv1D × 2 + Transformer → 1500 context vectors

Whisper's encoder applies two layers of **Conv1D downsampling** to the `(80, 3000)` log-mel, then feeds the result into a Transformer.

```
Conv1d(80 → d, kernel=3, padding=1, stride=1)   # 3000 → 3000
Conv1d(d  → d, kernel=3, padding=1, stride=2)   # 3000 → 1500
```

The two conv layers downsample by **2×** in total: stride=1 keeps 3000 frames, stride=2 compresses them to 1500.

### A geometric reading of the Conv1d output-length formula

Why does that happen? Let's understand Conv1d's stride parameter from the point of view of a **sliding window**.

**The basic idea**: a Conv1d kernel is like a small window that slides along the sequence, computing one output at each position.

```
input sequence:  [x0, x1, x2, x3, x4, x5, x6, x7, x8, ...]
                  ↑
                 a kernel_size=3 window: [x0, x1, x2]
                 → output y0 = conv([x0, x1, x2], kernel)

            with stride=1 (the window moves on 1 position at a time):
            [x0, x1, x2] → y0
               [x1, x2, x3] → y1
                  [x2, x3, x4] → y2
                  ...

            with stride=2 (the window moves on 2 positions at a time):
            [x0, x1, x2] → y0
                  [x2, x3, x4] → y1
                        [x4, x5, x6] → y2
                  ...
```

**What stride=2 does**: the window jumps 2 positions each time, so there are roughly half as many outputs as inputs — that's "downsampling".

**The formula**:
```
n_out = (n_in + 2*padding - kernel_size) // stride + 1
```

Let's check Whisper's settings with numbers (kernel=3, padding=1, stride=1 or 2):

```
Conv1 (stride=1, padding=1):
  n_out = (3000 + 2*1 - 3) // 1 + 1 = (3000 - 1) // 1 + 1 = 3000 ✓

Conv2 (stride=2, padding=1):
  n_out = (3000 + 2*1 - 3) // 2 + 1 = (3000 - 1) // 2 + 1 = 2999 // 2 + 1 = 1499 + 1 = 1500 ✓
```

**Why add padding=1?**

Padding adds 1 zero (or some other value) at each end of the input. Its job:
- Without padding, the kernel runs short of input at the edges (kernel=3 needs 3 inputs, but near the edge there may not be enough).
- With padding, the kernel can be applied even at the edges, so the sequence doesn't shrink quickly.

```
Conv1d without padding (kernel=3, stride=1):
  n_out = (n_in - kernel) // stride + 1 = (3000 - 3) // 1 + 1 = 2998

Conv1d with padding=1 (kernel=3, stride=1):
  n_out = (3000 + 2*1 - 3) // 1 + 1 = 3000  ← length unchanged
```

**Whisper's design choice**:
- Layer 1 uses stride=1: it keeps 3000 frames and just widens the receptive field.
- Layer 2 uses stride=2: building on that, it compresses straight down to 1500 frames.
- Why not stride=2 in both layers? That would give 3000 → 750, compressing away too much information.

Note: if both conv layers used stride=2 you'd get 3000 → 1500 → 750 — and that isn't Whisper. In Whisper's official code (openai/whisper) only the second layer uses stride=2, and the encoders of **every size**, from tiny to large, output a context sequence of **1500** steps, `(1500, d_model)`.

```
encoder output shape: (B, 1500, d_model)
  d_model: tiny=384, base=512, small=768, medium=1024, large=1280
```

Each Transformer layer contains **self-attention → a feed-forward network (FFN)**; the encoder uses no causal mask (every frame can see the whole acoustic context).


In [ ]:
# Demo: a minimal hand-written fragment of the Whisper encoder in PyTorch (just the conv downsampling)
import torch.nn as nn
import torch.nn.functional as F

d_model = 512  # Whisper-base
conv1 = nn.Conv1d(80, d_model, kernel_size=3, padding=1, stride=1)
conv2 = nn.Conv1d(d_model, d_model, kernel_size=3, padding=1, stride=2)

dummy_mel = torch.zeros(1, 80, 3000)   # (B, n_mels, T)
x1 = F.gelu(conv1(dummy_mel))          # (1, 512, 3000)
x2 = F.gelu(conv2(x1))                 # (1, 512, 1500)
print(f"input: {tuple(dummy_mel.shape)} → Conv1 → {tuple(x1.shape)} → Conv2 → {tuple(x2.shape)}")
print(f"the encoder continues with Transformer layers over {x2.shape[-1]} steps")


## 3. The decoder: an autoregressive (AR) Transformer + cross-attention → BPE tokens

The decoder is a standard **autoregressive causal Transformer**, similar in structure to GPT, but with an extra **cross-attention** layer whose keys and values come from the encoder output.

```
The order inside each decoder layer:
  1. self-attention (causal mask: only looks at tokens already generated)
  2. cross-attention (Q from the decoder, K/V from the encoder's 1500 steps)
  3. FFN
```

### The causal mask: why can't the decoder see "the future"?

Imagine you're producing live captions for a broadcast:

```
live audio stream: [hello] [world] [,] [nice] [to] [meet] ...
                   ↑ the word being processed now

your decoder should:
  ✅ use the words already said: [hello] (to help predict the current word)
  ❌ never "peek" at later words: [world], [,], [nice], ... (that leaks information the model won't have at inference time)
```

In Transformer self-attention, the **causal mask** achieves this with a "masking matrix":

```
decoder self-attention scores (before masking):
        T0  T1  T2  T3  T4
    T0 [1   2   3   4   5]   step 0 can see every step
    T1 [6   7   8   9  10]   step 1 can see every step too (including the future T2, T3, T4)
    T2 [11 12  13  14  15]   step 2 can see every step too...
    T3 [16 17  18  19  20]
    T4 [21 22  23  24  25]

after applying the causal mask:
        T0  T1  T2  T3  T4
    T0 [1  -∞  -∞  -∞  -∞]   step 0 sees only itself
    T1 [6   7  -∞  -∞  -∞]   step 1 sees T0 and T1
    T2 [11 12  13  -∞  -∞]   step 2 sees T0, T1, T2
    T3 [16 17  18  19  -∞]   step 3 sees T0, T1, T2, T3
    T4 [21 22  23  24  25]   step 4 sees everything

after softmax (-∞ → 0), the future steps' weights are automatically "zeroed".
```

**Encoder vs decoder**:
- **Encoder self-attention**: no causal mask; every step can "see" all 1500 steps (bidirectional context). The reason: the encoder processes **the complete audio**, so the information is symmetric.
- **Decoder self-attention**: causal mask; token i can only see itself and the tokens before it. The reason: decoding is **generated step by step**, and must not violate time order.
- **Decoder cross-attention**: no causal mask (with respect to the encoder); every decoding step can see all 1500 encoder steps. The reason: the encoder output is already fixed — there's no "future", only "the whole".

Vocabulary size: **51865** (multilingual model), using tiktoken BPE. Special tokens: `<|startoftranscript|>`, a language tag such as `<|es|>`, a task tag `<|transcribe|>`, and either `<|notimestamps|>` or timestamp tokens (`<|0.00|>` to `<|30.00|>`).

The decoding process:

```
[SOT] → [language] → [task] → [content token_1] → [token_2] → ... → [EOT]
```

At every decoding step, cross-attention "looks at" the whole 1500-step encoder output to decide which sound — which piece of a word — to emit next.

### Digging deeper: why is cross-attention better than the alternatives?

We've seen what cross-attention "looks like" (Q from the decoder, K/V from the encoder), and why the encoder can see all 1500 steps. But there's a more fundamental question: **why design it this way? Compared with other schemes that "look like they'd work too", what makes it better?**

#### Alternative 1: "glue" the encoder and decoder tokens together and use plain self-attention

Since Transformer self-attention works so well, why not just concatenate the encoder's 1500 vectors with the tokens the decoder has generated so far into one long sequence, run self-attention over the whole thing, and skip a separate "cross-attention" module?

The problem is **computation** and **reuse**. Decoding is **autoregressive, one token at a time** — generate the 1st word, the 2nd word… up to 448 tokens.

- **If you glue them together for self-attention**: every time the decoder generates a new token, in principle it has to re-concatenate "the encoder's 1500 steps + every token generated so far" and recompute self-attention. Sequence length = 1500 + T_dec, cost ∝ (1500 + T_dec)² — and **the information in those 1500 encoder steps never changes from start to finish**, so recomputing with them every time is pure waste.
- **The cross-attention approach**: the encoder runs **once**, computes 1500 K and V vectors, and **caches** them; when the decoder later generates its 2nd, 3rd, …, 448th token, it reuses that cached K/V directly without recomputing the encoder. Each step of cross-attention costs only `T_dec_current × 1500`, not `(1500+T_dec)²`.

An analogy: the encoder's 1500 steps are like a reference book you've already read and summarized (K/V) — you don't reread the whole book every time you write a word; you just **keep consulting the same summary**. That's the practical value of "K/V come from the encoder": the encoder computes once, and the decoder can "ask" it many times.

#### Alternative 2: look only at the top-k most relevant steps instead of all 1500

Another intuition: decoding one word usually needs only a short stretch of the audio (say 100–150 steps), so why not let the model look at just those steps and ignore the rest, saving compute?

The problem: **how do you know in advance which steps to look at?** That needs an extra "selection mechanism" to judge "relevance", and that selection is precisely the problem attention exists to solve. If the selection is hard (selected means visible, not selected means completely invisible), then as soon as it picks the wrong key frames, the model **permanently loses** that information — and the "pick top-k" step is usually not differentiable, so gradients can't flow through it to correct a bad choice.

Cross-attention's softmax is really a **"soft" top-k**: it still computes a similarity for all 1500 steps, but through softmax the truly relevant few naturally get weights close to 1 and the irrelevant ones are squashed close to 0. The effect is much like "only look at the most relevant steps", but the whole process is **continuous and differentiable**: during training the model learns for itself "where to look", nobody has to fix k by hand, and a single "wrong pick" never throws information away completely (small weights aren't forced to exactly zero, so a little "backup" channel of information remains).

**One-line summary**: cross-attention = the encoder computes once and is reused over and over (saves compute) + softmax is a trainable "soft selection" (no hand-picked set of steps, and no total information loss from a wrong pick). Together, these make it cheaper than "glued-together self-attention" and more flexible and reliable than "hand-picked top-k".

### Digging deeper: why divide by sqrt(d_k), and not by d_k or some other number?

The code below contains an unassuming-looking division:

```python
attn_weight = (Q @ K.transpose(-1,-2)) / (head_dim ** 0.5)
```

`head_dim ** 0.5` is $\sqrt{d_k}$ ($d_k$ is the dimension of each attention head; for Whisper-base it's 512/8=64). In the Transformer paper (*Attention Is All You Need*, 2017) this step is called **scaled dot-product attention** — the "scaled" in the name is exactly this division. Why must we divide at all? And why the square root, rather than $d_k$ itself or any old constant? Let's derive it from scratch.

#### Step 1: how does the "variance" of a dot product grow with dimension?

Q and K are both vectors of length $d_k$; the dot product multiplies their matching components pairwise and sums:

```
Q·K = q_1·k_1 + q_2·k_2 + ... + q_{d_k}·k_{d_k}
```

Recall variance from L27–L28: if $q_i$ and $k_i$ are independent random numbers with "mean 0 and variance 1" (close to common parameter initializations in neural networks), then each product $q_i k_i$ has mean 0 and variance about 1. The dot product is a sum of $d_k$ such terms — and **when you add independent random variables, their variances add too**:

```
Var(Q·K) ≈ d_k × 1 = d_k
```

In other words: **the larger the dimension $d_k$, the larger the dot product's "swing" (standard deviation = $\sqrt{d_k}$)**. At $d_k=64$ the dot product's standard deviation is about 8; at $d_k=512$ it grows to about 22.6.

#### Step 2: dot products that swing too widely make softmax "lock up"

Softmax turns a list of numbers into weights that add up to 1, but if the input numbers swing widely (some 60, some -50), softmax gives almost all the weight to the largest one and pushes the rest toward 0 — this is called **saturation**.

Why is saturation bad? Flip back to the chain rule in L24: backpropagation has to pass gradients back through softmax, and softmax's gradient contains a factor of the form $p_i(1-p_i)$ (where $p_i$ is one of the weights). If $p_i$ is already very close to 1 or 0, then $p_i(1-p_i)$ is very close to 0 — **the gradient can barely get back through**, and the model stops learning. The bigger the dimension, the wilder the unscaled dot products swing, and the worse this gets.

#### Step 3: why does dividing by $\sqrt{d_k}$ exactly "fix" it?

We want the scaled dot product's standard deviation to stay at about 1 no matter how big $d_k$ is (neither large nor small, so softmax's input sits in a "gentle" range):

```
Var(Q·K / c) = Var(Q·K) / c²  ≈  d_k / c²
```

To make this equal 1 we just need $c^2 = d_k$, i.e. $c = \sqrt{d_k}$. **That's why it's the square root and not $d_k$ itself** — divide by $d_k$ (instead of $\sqrt{d_k}$) and the scaled variance becomes $d_k / d_k^2 = 1/d_k$: the bigger the dimension, the *more* it shrinks, all the dot products crowd around 0, and the attention weights become almost "all equal" (softmax output tends to a uniform distribution) — so again the model can't learn "which steps matter more". So $\sqrt{d_k}$ is the one choice that keeps the scaled variance steady at 1 (independent of $d_k$) — no more, no less, exactly cancelling the growth of $d_k$ in the variance.

Below, code checks these three steps of the derivation, and shows what happens when you "don't scale" or "over-scale".

In [ ]:
# Demo: the larger d_k, the larger the dot-product variance; dividing by sqrt(d_k) holds the variance at 1
import numpy as np

np.random.seed(0)
print("Step 1 check: Var(Q·K) ≈ d_k")
print("-" * 60)
for d_k in [16, 64, 256, 1024]:
    dots = np.array([np.random.randn(d_k) @ np.random.randn(d_k) for _ in range(3000)])
    print(f"d_k={d_k:5d}  measured Var(Q·K)={dots.var():8.1f}  theory ≈ d_k={d_k:5d}  measured std={dots.std():.2f}  theory sqrt(d_k)={d_k**0.5:.2f}")

print()
print("Step 2 check: no scaling vs divide by sqrt(d_k) vs divide by d_k — does softmax 'saturate'?")
print("-" * 60)

def softmax(x):
    e = np.exp(x - x.max())
    return e / e.sum()

d_k = 512  # deliberately a large dimension so the effect is obvious (a real head_dim is usually 64)
q = np.random.randn(d_k)
keys = np.random.randn(6, d_k)  # suppose the current decoding step is compared with 6 encoder steps

raw_scores = keys @ q                       # no scaling
scaled_scores = raw_scores / np.sqrt(d_k)   # divide by sqrt(d_k): what Whisper/Transformers do
over_scaled = raw_scores / d_k              # counter-example: divide by d_k itself

print("raw dot products raw_scores :", np.round(raw_scores, 1))
print("after dividing by sqrt(d_k) :", np.round(scaled_scores, 2))
print("after dividing by d_k (over):", np.round(over_scaled, 3))

print("\nsoftmax(no scaling)   :", np.round(softmax(raw_scores), 3), " ← almost one weight ≈1, the rest ≈0 (saturated, vanishing gradients)")
print("softmax(/ sqrt(d_k))  :", np.round(softmax(scaled_scores), 3), " ← clear winners and losers, but not extreme")
print("softmax(/ d_k)        :", np.round(softmax(over_scaled), 3), " ← weights almost uniform, no clear winners (over-scaled)")


In [ ]:
# Demo: the shape relationships in cross-attention
B, T_enc, T_dec = 1, 1500, 20  # 1500 encoder steps; decoding has reached the 20th token
d_model, n_heads = 512, 8
head_dim = d_model // n_heads

Q = torch.randn(B, T_dec, d_model)    # from the decoder's current state (1, 20, 512)
K = torch.randn(B, T_enc, d_model)    # from the encoder output (1, 1500, 512)
V = torch.randn(B, T_enc, d_model)    # from the encoder output (1, 1500, 512)

print("=" * 60)
print("Working out the dimensions of cross-attention")
print("=" * 60)
print(f"Q shape (from the decoder): {tuple(Q.shape)}")
print(f"K shape (from the encoder): {tuple(K.shape)}")
print(f"V shape (from the encoder): {tuple(V.shape)}")

# step 1: transpose K
K_T = K.transpose(-1, -2)
print(f"\nK.transpose(-1, -2): {tuple(K.shape)} → {tuple(K_T.shape)}")
print(f"  (swaps the positions of T_enc=1500 and d_model=512)")

# step 2: Q @ K^T
print(f"\nQ @ K.T matrix multiplication:")
print(f"  Q:    {tuple(Q.shape)}")
print(f"  K.T:  {tuple(K_T.shape)}")
print(f"  ignore the batch dimension and look at the last two:")
print(f"    (T_dec, d_model) × (d_model, T_enc)")
print(f"    (20, 512) × (512, 1500)")
print(f"    by the matrix-multiplication rule (m,n) × (n,p) = (m,p):")
print(f"    (20, 512) × (512, 1500) = (20, 1500)")

attn_weight = (Q @ K.transpose(-1,-2)) / (head_dim ** 0.5)
print(f"\nattention weight shape: {tuple(attn_weight.shape)}")
print(f"  meaning: (batch, T_dec, T_enc) = (1, 20, 1500)")
print(f"  row i, column j = the raw similarity of decoding step i to encoder step j")

# step 3: softmax
attn_weight_norm = torch.softmax(attn_weight, dim=-1)
print(f"\nweights after softmax: {tuple(attn_weight_norm.shape)}")
print(f"  the 1500 numbers in each row add up to 1.0 (a probability distribution)")
print(f"  for example, the sum of the first 10 values of row 0 = {attn_weight_norm[0, 0, :10].sum():.4f}")

# step 4: weighted sum
out = attn_weight_norm @ V
print(f"\nattention output = weights @ V:")
print(f"  attn_weight: {tuple(attn_weight_norm.shape)}")
print(f"  V:           {tuple(V.shape)}")
print(f"  (1, 20, 1500) × (1, 1500, 512) = (1, 20, 512)")
print(f"  out shape:   {tuple(out.shape)}")
print(f"\nInterpretation: each of the 20 decoding steps \"sees\" a weighted average of the encoder's 1500 steps.")
print(f"      The weights are set by how similar that step's Q is to every encoder K.")
print("=" * 60)

# Echoing the derivation in the markdown above: why we divide by sqrt(head_dim) here and not some other number
# Note: attn_weight above is a dot product over the full d_model=512-dim vectors (to keep the focus on the
# shape derivation, we didn't actually split into "heads" first). Here we slice out the head_dim=64 dims the first head would use, matching the markdown derivation.
print("\nWhy divide by sqrt(head_dim)? Check it on a \"single-head\" slice:")
Q_head = Q[..., :head_dim]           # simulate slicing out the first head's 64 dims
K_head = K[..., :head_dim]
raw_scores_head = Q_head @ K_head.transpose(-1, -2)
print(f"  head_dim = {head_dim}  → theoretical sqrt(head_dim) = {head_dim ** 0.5:.2f}")
print(f"  std of the unscaled raw_scores: {raw_scores_head.std():.2f}  (in theory ≈ sqrt(head_dim) = {head_dim ** 0.5:.2f})")
print(f"  std after dividing by sqrt(head_dim): {(raw_scores_head / (head_dim ** 0.5)).std():.2f}  (should be ≈ 1)")
print("  → scaling pulls the std from sqrt(head_dim) back to ≈1, so however big head_dim is, the \"steepness\" of softmax's input stays about the same.")
print("  (The attn_weight in the full computation above uses the full 512-dim dot product — this lesson's simplified shape demo;")
print("   real multi-head attention first splits Q/K/V into n_heads pieces of head_dim dims each, then scales each one like this.)")


### Matrix multiplication and transpose on 3D tensors, in detail

The matrix multiplication you learned at school was 2D: `(m, n) × (n, p) = (m, p)`. But deep learning is full of 3D and higher-dimensional tensors — how should you understand an operation like `Q @ K.transpose(-1, -2)`?

**The key principle: PyTorch leaves the "batch" dimensions alone and only does matrix multiplication on the last ones.**

**Example**: say there are 2 batches, each with a (3, 4) matrix and a (4, 5) matrix. How do they multiply?

```python
A = torch.randn(2, 3, 4)  # (batch=2, rows=3, cols=4)
B = torch.randn(2, 4, 5)  # (batch=2, rows=4, cols=5)

C = A @ B  # what's the result?
```

**Intuition**: "ignore" the batch dimension and look only at the last two:
- batch 0: (3, 4) × (4, 5) = (3, 5)
- batch 1: (3, 4) × (4, 5) = (3, 5)
- so `C.shape = (2, 3, 5)`

**Check**: `C[0] = A[0] @ B[0]` and `C[1] = A[1] @ B[1]`.

---

**Applied to cross-attention**: in Whisper's decoder we have:
- Q (query) from the decoder: shape `(B, T_dec, d_model) = (1, 20, 512)`
- K (key) from the encoder: shape `(B, T_enc, d_model) = (1, 1500, 512)`
- and we want the attention weights: `Q @ K^T`

But K is `(1, 1500, 512)`; we need it as `(1, 512, 1500)` to multiply with Q. How do we turn it?

**What transpose means**:

```python
K = torch.randn(1, 1500, 512)  # (batch, T_enc, d_model)
K.transpose(-1, -2)            # (batch, d_model, T_enc) = (1, 512, 1500)
```

`transpose(-1, -2)` means "swap the second-to-last and the last dimensions". For a 3D tensor (B, T, D) that swaps T and D, giving (B, D, T).

**Now they can be multiplied**:
```
Q:              (1, 20, 512)
K.T:            (1, 512, 1500)

Q @ K.T = ?
```

Following the "ignore the batch" principle:
- the batch dimension stays put: (20, 512) × (512, 1500) = (20, 1500)
- so `(Q @ K.T).shape = (1, 20, 1500)`

That is the **attention weight matrix**: each row is a decoding step, each column an encoder step, and element [i, j] says "how much decoding step i attends to encoder step j".

**Why transpose(-1, -2) rather than transpose(1, 2)?**

`transpose(-1, -2)` is more general across tensors of different dimensions:
- 3D: `(B, T, D).transpose(-1, -2)` → `(B, D, T)` ✓
- 4D: `(B, H, T, D).transpose(-1, -2)` → `(B, H, D, T)` ✓ (multi-head attention)
- `transpose(1, 2)` would be wrong in 4D



## 4. The multitask head: one decoder, four kinds of output

What makes Whisper distinctive isn't only the encoder–decoder architecture, but that **one model uses special tokens as conditioning to perform four different tasks**:

| Task | Head output | Conditioning tokens |
|------|--------|-----------|
| Speech recognition (transcribe) | a BPE text sequence | `[SOT][lang][transcribe][notimestamps]` |
| Speech translation (translate) | an English BPE sequence | `[SOT][lang][translate][notimestamps]` |
| Language identification (language-ID) | lang token probabilities | `[SOT]` → take the next-token distribution |
| Recognition with timestamps | BPE interleaved with timestamp tokens | `[SOT][lang][transcribe]` (no notimestamps) |

### How do the prefix tokens "direct" the model to do different tasks?

**The core idea**: Whisper doesn't tell the tasks apart with "two different models" or "different architectural branches". It uses **one shared embedding and one set of parameters**, and the meaning of the prefix tokens is built in **during training**.

**An analogy**: imagine you have a smart assistant (the model) with an "instruction card" system:

```
Scenario A: I say "What's the weather like?"
  → you query a weather API → you answer with the weather

Scenario B: I say "Translate this into English", then give you a Spanish sentence
  → you know to translate → you do the translation task

The model works the same way:
  • it sees prefix = [SOT, LANG_ES, TRANSCRIBE, NO_TIMESTAMPS]
    → "Ah, this is a 'transcribe' task: turn the Spanish speech into Spanish text"

  • it sees prefix = [SOT, LANG_ES, TRANSLATE, NO_TIMESTAMPS]
    → "Ah, this is a 'translate' task: turn the Spanish speech into English"
```

**What happens during training**:

Whisper is trained on **more than 680,000 hours** of multilingual audio, with labels for a mixture of tasks:

```
Data example 1 (transcribe):
  input audio:   [Spanish speech...]
  prefix:        [SOT, LANG_ES, TRANSCRIBE, NO_TIMESTAMPS]
  target tokens: [hola, mundo]
  loss:          CrossEntropy(prediction, target)

Data example 2 (translate):
  input audio:   [the same Spanish speech...]
  prefix:        [SOT, LANG_ES, TRANSLATE, NO_TIMESTAMPS]
  target tokens: [hello, world]  # the English translation
  loss:          CrossEntropy(prediction, target)

Data example 3 (language identification):
  input audio:   [Spanish speech...]
  prefix:        [SOT]
  target token:  LANG_ES  # predict only the language, generate no content
```

After a great deal of training, the model has learned that:
- the **TRANSCRIBE token**'s embedding activates the "recognize and transcribe" circuitry
- the **TRANSLATE token**'s embedding activates the "recognize and translate into English" circuitry
- the **LANG_ES token** tells the model "the input is Spanish" (shifting the softmax distribution so Spanish words become more likely)

**Key point: all of this happens inside the Transformer's self-attention and cross-attention**. Through attention, the prefix tokens "broadcast" their information to every later token:

```
decoder self-attention weights (simplified, unnormalized illustration; causal, so no token sees the future):
              SOT  LANG  TRANSCRIBE  NO_TS  CONTENT1  CONTENT2
    SOT       [ 1    0      0         0      0         0 ]  (SOT attends to itself)
    LANG      [ 0.3  1      0         0      0         0 ]  (language token attends to SOT and itself)
    TRANSCRIBE[ 0.2  0.3    1         0      0         0 ]  (task token attends to the conditions before it)
    NO_TS     [ 0.1  0.2    0.3       1      0         0 ]  (format token attends to the whole prefix)
    CONTENT1  [ 0.1  0.15   0.25      0.2    1         0 ]  (the content word sees the whole prefix)
    CONTENT2  [ 0.1  0.1    0.2       0.15   0.3       1 ]  (later content words see the whole history)
```

The attention weights tell us: **every later token is generated from a "weighted average" that includes the prefix tokens' information**. For example, "CONTENT1"'s representation is influenced 25% by the "TRANSCRIBE" embedding and 15% by the "LANG_ES" embedding. That's how the difference between "translate" and "transcribe" in the prefix steers the generation of the content tokens.

---

**The key design**: every task shares the same **output projection layer** (`Linear(d_model, vocab_size)`); the tasks are told apart entirely by the prefix token sequence. In other words, at training time you simply mix all four kinds of data together, and at inference time you switch tasks by appending different special tokens after `[SOT]`.

```
token prefix at decoding time (transcribe, for example):
  [SOT=50258] [es=50262] [transcribe=50359] [notimestamps=50363] → content tokens...
```

vocab_size = **51865** (50257 BPE + 1608 special tokens), of which 1501 are timestamp tokens (`<|0.00|>` to `<|30.00|>`, in steps of 0.02 s, both ends included).


In [ ]:
# Demo: conditioning the multitask head with tokens
# special token IDs (from the openai/whisper multilingual tokenizer)
SOT         = 50258   # start of transcript
LANG_ES     = 50262   # Spanish
TRANSCRIBE  = 50359   # transcription task
TRANSLATE   = 50358   # translation task
NO_TIMESTAMPS = 50363 # no timestamps in the output (50362 is <|nospeech|>)

# prefix sequences for three tasks
task_prefixes = {
    "transcribe (Spanish)": [SOT, LANG_ES, TRANSCRIBE, NO_TIMESTAMPS],
    "translate → English":  [SOT, LANG_ES, TRANSLATE, NO_TIMESTAMPS],
    "language ID":          [SOT],            # only look at the next token's distribution
}
for task, prefix in task_prefixes.items():
    print(f"{task}: prefix tokens = {prefix}")

# shape of the output projection layer
import torch.nn as nn
vocab_size = 51865
d_model = 512  # Whisper-base
output_proj = nn.Linear(d_model, vocab_size, bias=False)
print(f"\nshared output projection: Linear({d_model}, {vocab_size}) = {sum(p.numel() for p in output_proj.parameters()):,} parameters")
print("every task shares the same output_proj; switching tasks only means changing the prefix tokens.")

## 5. ✏️ Implement `whisper_preprocess(wav, sr=16000)`

**Signature**:
```python
def whisper_preprocess(wav: np.ndarray, sr: int = 16000) -> torch.Tensor:
    # returns: a (1, 80, 3000) float32 tensor (batch_size=1; the channel dimension of 1 is omitted)
```

**A 4-step implementation plan**:

| Step | What to do | Parameters |
|---|---|---|
| 1 | `mel_spectrogram(wav, sr, ...)` | n_fft=400, hop=160, n_mels=80 |
| 2 | Whisper-style log normalization | `log10` → clip to `max - 8` → `(x + 4) / 4` (not `power_to_db`) |
| 3 | pad/truncate → a fixed 3000 frames | after `mel.T` the shape is (80, T); T may be ≠3000 (30 s gives exactly 3001) |
| 4 | `torch.tensor(...).float().unsqueeze(0)` | (80, 3000) → (1, 80, 3000) |

**Acceptance criteria**:
- `whisper_preprocess(np.zeros(16000*30)).shape == (1, 80, 3000)`
- `dtype == torch.float32`
- audio under 30 seconds is padded to 3000 frames; longer audio is truncated

### Extra: why does the encoder output 1500 vectors?

You might ask: once the conv layers have brought 3000 frames down to 1500, why don't the Transformer layers change the sequence length any further?

**The answer**: the Transformer's self-attention and feed-forward network both preserve sequence length — the FFN works **position-wise**, and self-attention produces one output for every input position.

```
input:  (B, T, d_model) = (1, 1500, 512)
  ↓ self-attention (every position "looks at" all T=1500 steps, but there are still 1500 output vectors)
  ↓ feed-forward network (each position processed independently)
output: (B, T, d_model) = (1, 1500, 512)
```

**Why do the convolutions downsample while the Transformer doesn't?**

- The **convolutions** do "feature extraction" and "denoising": the raw 3000-frame mel spectrogram is very redundant, with little difference between neighboring frames. The stride=2 conv layer removes that redundancy.
- The **Transformer** does "global association" and "context aggregation": 1500 "sparser" feature frames are enough to cover 30 seconds of audio (each Transformer step corresponds to 20 ms of audio on average). Self-attention lets each step see the context of all 1500 steps — enough to work out "which phoneme this is".

**What does each Transformer output vector represent?**

```
encoder output: (1, 1500, 512) — 1500 vectors
      ↓
      vector i (the output at step i) represents:
      • time alignment: roughly the point i × 20ms into the audio (because hop=160 is 10ms,
                                    and the two conv layers multiply that by 2, giving 20ms)
      • meaning: a mixed representation of the audio's "acoustic features" at this moment (pitch, formants, silence and so on)
                and "global context" (information about the surrounding phonemes)

      This 512-dim vector is called a "context vector" or "acoustic embedding"
```

**The decoder's job**:

The decoder reads these 1500 vectors and, through cross-attention, "asks" each step "which stretch of audio does this token belong to?". For example:
- when decoding the word "hola", cross-attention might give high weight to encoder steps 100–150
- when decoding the word "mundo", it might give high weight to steps 200–250
- ...and so on

So the Transformer keeps 1500 steps in order to **leave enough time resolution** for the cross-attention that follows to "pinpoint" the stretch of audio each token belongs to.



In [ ]:
def whisper_preprocess(wav: np.ndarray, sr: int = 16000) -> torch.Tensor:
    """
    Convert a raw waveform into the log-mel feature tensor Whisper expects.

    Parameters
    ----------
    wav : np.ndarray, shape (T,)
        Mono PCM waveform, at sample rate sr.
    sr : int
        Sample rate; Whisper's default is 16000 Hz.

    Returns
    -------
    torch.Tensor, shape (1, 80, 3000), dtype=float32
    """
    # ✏️ TODO: step 1 — call aurora's mel_spectrogram with n_fft=400, hop_length=160, n_mels=80
    #          note that it returns shape (n_frames, 80), so transpose with .T

    # ✏️ TODO: step 2 — log normalization (Whisper style)
    #          log_mel = log10(max(mel, 1e-10))
    #          log_mel = max(log_mel, log_mel.max() - 8.0)
    #          log_mel = (log_mel + 4.0) / 4.0

    # ✏️ TODO: step 3 — pad or truncate to 3000 frames (axis=1)

    # ✏️ TODO: step 4 — convert to a torch.float32 tensor, unsqueeze(0)
    raise NotImplementedError("whisper_preprocess: implement this function following steps 1-4")


In [ ]:
try:
    # check: 30 seconds of silence
    feat = whisper_preprocess(np.zeros(16000 * 30))
    assert feat.shape == (1, 80, 3000), f"wrong shape: {feat.shape}"
    assert feat.dtype == torch.float32, f"wrong dtype: {feat.dtype}"
    print(f"✅ shape={tuple(feat.shape)}, dtype={feat.dtype}")
    print(f"✅ value range: [{feat.min():.3f}, {feat.max():.3f}] (a constant -1.5 for pure silence)")

    # check: short audio is zero-padded automatically
    feat_short = whisper_preprocess(np.zeros(16000 * 5))  # 5 seconds
    assert feat_short.shape == (1, 80, 3000), f"zero-padding short audio failed: {feat_short.shape}"
    print(f"✅ 5 s of short audio zero-padded → shape={tuple(feat_short.shape)}")

    # check: long audio is truncated automatically
    feat_long = whisper_preprocess(np.zeros(16000 * 60))  # 60 seconds
    assert feat_long.shape == (1, 80, 3000), f"truncating long audio failed: {feat_long.shape}"
    print(f"✅ 60 s of long audio truncated → shape={tuple(feat_long.shape)}")
except (NotImplementedError, TypeError):
    print('⚠️ whisper_preprocess is not implemented yet — complete steps 1-4, then run this check')


## 6. Parameter experiment: how Aurora's log-mel differs from official Whisper's

Compare `aurora.audio.mel.mel_spectrogram` with the official openai/whisper `log_mel_spectrogram` (`whisper/audio.py`) and find the differences in the following parameters:

| Parameter | Aurora default | Official Whisper |
|------|-------------|-------------|
| `n_fft` | 1024 | **400** |
| `hop_length` | `n_fft // 4 = 256` | **160** |
| `n_mels` | 80 | 80 (same) |
| `fmin` | 0.0 | **0.0** (same) |
| `fmax` | `sr / 2` | **sr / 2** (same) |
| log normalization | `power_to_db` (dB, `10*log10`) | **`log10` + clip(-8) + /4 normalization** |
| output shape | `(n_frames, n_mels)` | **`(n_mels, n_frames)`** |

**What to expect**:
- With Aurora's default parameters (`n_fft=1024, hop=256`), 30 s of audio gives `(1876, 80)` → `(80, 1876)` after transposing — short of 3000 frames, so even after zero-padding its **time resolution** differs from the official result (and its frequency resolution is finer).
- With Whisper's parameters (`n_fft=400, hop=160`), 30 s gives `(3001, 80)` — center padding adds 1 frame, and once the last frame is dropped it lines up with the official 3000 frames. (The frames line up, but the values are not identical: `mel_spectrogram` computes the 400-point frame on a zero-padded 512-point FFT grid and uses the HTK mel scale, whereas Whisper uses a true 400-point FFT and the Slaney mel scale — see REPORT.md and `aurora.audio.whisper_frontend.log_mel_whisper`.)
- The normalization differs: `power_to_db` outputs dB values (e.g. -80 to 0 dB), while Whisper's normalized values lie roughly in `[-1, 1]`.


In [ ]:
# Experiment 1: frame counts with Aurora's default parameters vs Whisper's parameters
wav_30s = np.zeros(16000 * 30)

# Aurora default (n_fft=1024, hop=256)
mel_aurora_default = mel_spectrogram(wav_30s, 16000, n_fft=1024, hop_length=256, n_mels=80)
print(f"Aurora defaults  → mel shape: {mel_aurora_default.shape}  (frames={mel_aurora_default.shape[0]})")

# Whisper parameters (n_fft=400, hop=160)
mel_whisper = mel_spectrogram(wav_30s, 16000, n_fft=400, hop_length=160, n_mels=80)
print(f"Whisper params   → mel shape: {mel_whisper.shape}  (frames={mel_whisper.shape[0]})")

# Experiment 2: comparing log normalizations
mel_raw = mel_spectrogram(wav_30s[:16000], 16000, n_fft=400, hop_length=160, n_mels=80)  # 1 second

# Aurora power_to_db style
db = power_to_db(mel_raw)
print(f"\npower_to_db range: [{db.min():.1f}, {db.max():.1f}] dB")

# Whisper style
log_mel = np.log10(np.maximum(mel_raw, 1e-10))
log_mel = np.maximum(log_mel, log_mel.max() - 8.0)
log_mel = (log_mel + 4.0) / 4.0
print(f"Whisper normalization range: [{log_mel.min():.3f}, {log_mel.max():.3f}]")
print("\n✅ Conclusion: to feed the official Whisper encoder you must use n_fft=400, hop=160, and Whisper-style normalization")


## 🎯 The payoff later

The data flow you took apart yourself today — **log-mel → encoder → cross-attention decoder → multitask heads** — will keep coming back. It's worth memorizing every shape now:

- **L71**: how the decoder generates "one token at a time" — greedy vs beam search, acting on exactly the decoder output distribution drawn in this lesson.
- **L72**: LoRA fine-tuning injects low-rank branches only into `q_proj`/`v_proj`, and those two projections live in the attention layers you just dissected; without understanding the architecture you wouldn't know where to plug them in.
- **L75**: visualizing cross-attention heatmaps — "which audio frames is the decoder's k-th token looking at?" — is this lesson's shape relationship drawn out on real data.

In one line: everything after Whisper — **fine-tuning, decoding, visualization** — is built on the data-flow diagram you worked out today.


## Wrap-up

This lesson implemented `whisper_preprocess(wav)`, turning 30 seconds of audio into the `(1, 80, 3000)` input Whisper needs; this step genuinely connects Aurora DSP to the Whisper encoder. It relies on `aurora.audio.mel.mel_spectrogram`, lined up with Whisper's frame parameters by setting `n_fft=400, hop_length=160`, plus Whisper-style log normalization (`log10 → clip → /4`). The next lesson (L71) implements Whisper's two decoding strategies: greedy decoding (take the most probable token at each step) and beam search with a width of 2, comparing the quality of the sequences they produce on a toy language model.

## ✏️ Closed-book derivation check cell — the Whisper data-flow diagram

**Rule: close every cell above and complete the derivation below from memory.**

**Question**: Whisper-small's parameters:
- Input: 30 seconds of audio, 80 mel bins, 10 ms hop → 3000 frames
- Encoder: 12 Transformer layers, d_model=768, 12 heads (64 dims each)
- Decoder: takes a BOS token as input and generates autoregressively

Draw the data-flow diagram and label the **tensor shape** at every step:

| Step | Operation | Output shape |
|------|------|---------|
| 1 | log-mel input | (batch, 80, 3000) |
| 2 | Conv1 (kernel=3, stride=1, out=768) | (batch, ___, ___) |
| 3 | Conv2 (kernel=3, stride=2, out=768) | (batch, ___, ___) |
| 4 | Encoder output | (batch, ___, ___) |
| 5 | Decoder cross-attention K/V come from | the output of step ___ |

(Fill in the table here...)

In [ ]:
# Verify: compute the output shapes of Whisper's conv layers (no model weights needed)
import numpy as np

def conv1d_out(L_in, kernel, stride=1, padding=1):
    return (L_in + 2*padding - kernel) // stride + 1

T = 3000   # 3000 mel frames
after_conv1 = conv1d_out(T, kernel=3, stride=1, padding=1)  # keeps the length
after_conv2 = conv1d_out(after_conv1, kernel=3, stride=2, padding=1)  # downsamples 2x

# Whisper-small encoder sequence length = 1500
assert after_conv1 == 3000, f"Conv1 output {after_conv1}, expected 3000"
assert after_conv2 == 1500, f"Conv2 output {after_conv2}, expected 1500"
print(f"Conv1 output sequence length: {after_conv1}")
print(f"Conv2 output sequence length: {after_conv2}")
print(f"✅ Whisper dimensions worked out correctly: (batch, 768, 1500) enters the Transformer encoder")

### Extra: when does decoding stop? What T_dec really means

In the cross-attention demo above we set `T_dec = 20`, meaning the decoder has generated 20 tokens. But is that "20" fixed? A sentence might be 10 tokens, or 100. So when does decoding stop?

**The answer**: the decoder generates a special **"end token" (EOT, End-Of-Transcript)**, and decoding stops when it produces that token.

```
The decoding process (sketch of the greedy strategy):

  Step 0 (initialization): input prefix [SOT, LANG_ES, TRANSCRIBE, NO_TIMESTAMPS]
                           the encoder outputs 1500 vectors (unchanged from here on)

  Step 1: the decoder autoregressively generates content token 1
         - prefix + generated content = [SOT, LANG_ES, TRANSCRIBE, NO_TIMESTAMPS] (no content yet)
         - computed through self-attention and cross-attention
         - outputs a softmax distribution (vocab_size=51865,)
         - take the most probable token ID → e.g. "hola" (ID=4321)

  Step 2: go on to generate token 2
         - prefix + generated content = [..., hola]
         - through the decoder again (T_dec is now 5: 4 prefix tokens + 1 generated content token)
         - outputs a distribution
         - take the most probable token → e.g. "mundo" (ID=5678)

  Steps 3, 4, ..., N: repeat the process above

  Step N: the model outputs EOT (ID=50257)
         → decoding stops, output [hola, mundo] (prefix and EOT removed)
```

**What T_dec actually means**:

In a Transformer decoder, `T_dec` isn't "a fixed 20" — it's **the length of the sequence generated so far (including the prefix)**.

```
start:        T_dec = len(prefix) = 4  ([SOT, LANG_ES, TRANSCRIBE, NO_TIMESTAMPS])
after step 1: T_dec = 5  (prefix + 1 content token)
after step 2: T_dec = 6  (prefix + 2 content tokens)
...
after step N: T_dec = len(prefix) + N
```

**Implementation details**:

```python
def whisper_decode_greedy(encoder_output, prefix_tokens, max_tokens=448):
    """
    encoder_output: (1, 1500, 512)  — the encoder output (fixed)
    prefix_tokens: [SOT, LANG, TASK, NO_TIMESTAMPS]  — the prefix (fixed)
    max_tokens: the most content tokens to generate (prevents generating forever)
    """
    generated = prefix_tokens.copy()  # initialize

    for step in range(max_tokens):
        # the sequence generated so far
        current_seq = torch.tensor(generated).unsqueeze(0)  # (1, T_dec)

        # through the decoder (T_dec = len(generated) keeps changing)
        logits = decoder(current_seq, encoder_output, causal_mask=True)
        # logits: (1, T_dec, vocab_size)

        # take only the output at the last step (the newest position)
        next_token_logits = logits[0, -1, :]  # (vocab_size,)

        # greedy choice: take the most probable token
        next_token = next_token_logits.argmax()

        if next_token == EOT_TOKEN_ID:
            break

        generated.append(next_token)

    return generated[len(prefix_tokens):]  # return the content part (prefix removed)
```

**Why does the implementation "take only the output at the last step"?**

Look at this line:
```python
next_token_logits = logits[0, -1, :]  # only the last step
```

Although the decoder outputs logits for all T_dec positions at once, we only care about **the distribution for the newest token**. The reason:
- the logits at earlier positions (tokens already generated) aren't needed any more (those tokens are already decided)
- only "the next position to generate" (step T_dec) needs to be sampled

**Summary**:
- `T_dec` isn't fixed; it grows as generation proceeds: 4 → 5 → 6 → ...
- when decoding stops: when the model outputs the EOT token
- to prevent infinite loops you normally set `max_tokens` (e.g. 448, the decoder's maximum text context length — the most tokens one 30-second window can hold)



In [ ]:
# ✏️ Self-review for this lesson
l70_review = {
    "whisper_3stage_understood":  None,  # Do you remember Whisper's stages: log-mel→Encoder→Decoder→BPE? True/False
    "frame_count_3000":           None,  # Do you understand how 30s@16kHz with hop=160 gives 3000 frames? True/False
    "cross_attention_understood": None,  # Do you understand decoder cross-attention: Q from the decoder, K/V from the encoder? True/False
    "whisper_preprocess_impl":    None,  # Is whisper_preprocess() correct, passing the shape/dtype checks? True/False
    "whiteboard_passed":          None,  # Did you clear the whiteboard challenge deriving the Whisper data-flow diagram? True/False
}

unfilled = [k for k, v in l70_review.items() if v is None]
assert not unfilled, f'Not filled in yet: {unfilled}'
weak = [k for k, v in l70_review.items() if v is False]
if weak:
    print(f'⚠️  Needs more work: {weak}')
else:
    print('✅ L70 all clear! On to L71: Whisper decoding')

---

→ **Next lesson** · [L71 · Whisper decoding](L71_whisper_decoding.ipynb)

> Next lesson covers **Whisper decoding**: greedy decoding and beam search, from principle to implementation.